In [ ]:
!pip install boto3 requests pandas --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 4.2 MB/s eta 0:00:00


In [ ]:

import requests
import pandas as pd
import time
from pathlib import Path

API_KEY = "e71d14b2b3615acb0f9243c48d526d9a9acab7578d9c65d476c592562f61c3e9"  # <-- replace
BASE_URL = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}

OUT_PATH = Path("./data/raw/openaq_india_locations.csv")


def get_india_locations() -> pd.DataFrame:
    """Paginate through all OpenAQ locations tagged with country=IN."""
    all_locations = []
    page = 1
    limit = 1000  # max page size for locations endpoint

    while True:
        params = {
            "iso": "IN",       # ISO country code filter for India
            "limit": limit,
            "page": page,
        }
        resp = requests.get(f"{BASE_URL}/locations", headers=HEADERS, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        results = data.get("results", [])

        if not results:
            break

        for loc in results:
            all_locations.append({
                "location_id": loc.get("id"),
                "name": loc.get("name"),
                "locality": loc.get("locality"),
                "latitude": loc.get("coordinates", {}).get("latitude"),
                "longitude": loc.get("coordinates", {}).get("longitude"),
                "provider": loc.get("provider", {}).get("name"),
                "datetime_first": loc.get("datetimeFirst", {}).get("utc") if loc.get("datetimeFirst") else None,
                "datetime_last": loc.get("datetimeLast", {}).get("utc") if loc.get("datetimeLast") else None,
                "parameters": [p.get("parameter", {}).get("name") for p in loc.get("sensors", [])],
            })

        print(f"Page {page}: {len(results)} locations (total so far: {len(all_locations)})")
        page += 1
        time.sleep(0.5)  # polite pacing even though metadata calls are cheap

        if len(results) < limit:
            break  # last page

    return pd.DataFrame(all_locations)


if __name__ == "__main__":
    df = get_india_locations()
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT_PATH, index=False)
    print(f"\nSaved {len(df)} India locations to {OUT_PATH}")

    # Quick sanity checks worth running manually after this:
    print(f"\nProviders found: {df['provider'].value_counts()}")
    print(f"\nDate coverage sample:\n{df[['name', 'datetime_first', 'datetime_last']].head(10)}")

    # FILTER TIP: You likely want provider == 'CPCB' specifically (some India
    # locations on OpenAQ come from other providers). Also check
    # datetime_first / datetime_last against your 2022-2025 target window -
    # stations with datetime_last well before 2022 or datetime_first well
    # after 2025 aren't useful for your period and can be dropped here.


Page 1: 758 locations (total so far: 758)

Saved 758 India locations to data/raw/openaq_india_locations.csv

Providers found: provider
CPCB                  573
caaqm                  76
N/A                    47
AirGradient            44
AirNow                  4
Clean Air Catalyst      3
StateAir Kolkata        2
data.gov.in             2
StateAir Chennai        1
StateAir Hyderabad      1
Spartan                 1
StateAir NewDelhi       1
HabitatMap              1
Clarity                 1
SPARTAN Network         1
Name: count, dtype: int64

Date coverage sample:
                                           name        datetime_first  \
0                          SPARTAN - IIT Kanpur                  None   
1  Delhi Technological University, Delhi - CPCB  2016-11-02T19:00:00Z   
2                                   IGI Airport                  None   
3                                   Civil Lines                  None   
4                       R K Puram, Delhi - DPCC  2016-02-05T1

In [ ]:

"""
Step 2 of 2: Bulk-download historical CPCB measurement data from OpenAQ's
public S3 archive, for the location IDs found in Step 1.

No AWS account needed - the bucket is public and this uses unsigned requests.
No rate limits like the API - this is plain S3 file storage.

Setup:
  pip install boto3 --break-system-packages
"""

import boto3
from botocore import UNSIGNED
from botocore.config import Config
import pandas as pd
import gzip
import io
from pathlib import Path

BUCKET = "openaq-data-archive"
YEARS = [2022, 2023, 2024, 2025]

LOCATIONS_PATH = Path("./data/raw/openaq_india_locations.csv")
RAW_CACHE_DIR = Path("./data/raw/openaq_s3_cache")
OUT_PATH = Path("./data/raw/openaq_india_2022_2025.csv")

RAW_CACHE_DIR.mkdir(parents=True, exist_ok=True)

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))


def list_files_for_location_year(location_id: int, year: int) -> list[str]:
    """List all daily csv.gz files under a location/year prefix."""
    prefix = f"records/csv.gz/locationid={location_id}/year={year}/"
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys


def download_and_read(key: str) -> pd.DataFrame | None:
    """Download one gzipped CSV file and return it as a DataFrame."""
    cache_file = RAW_CACHE_DIR / key.replace("/", "_")
    if cache_file.exists():
        try:
            return pd.read_csv(cache_file)
        except Exception:
            pass  # corrupted cache, refetch below

    try:
        obj = s3.get_object(Bucket=BUCKET, Key=key)
        raw_bytes = obj["Body"].read()
        with gzip.GzipFile(fileobj=io.BytesIO(raw_bytes)) as gz:
            df = pd.read_csv(gz)
        df.to_csv(cache_file, index=False)  # cache locally so reruns are cheap
        return df
    except Exception as e:
        print(f"  FAILED: {key}: {e}")
        return None


def bulk_download(location_ids: list[int]) -> pd.DataFrame:
    all_dfs = []
    total = len(location_ids)

    for i, loc_id in enumerate(location_ids, 1):
        print(f"[{i}/{total}] location_id={loc_id}")
        file_count = 0

        for year in YEARS:
            keys = list_files_for_location_year(loc_id, year)
            for key in keys:
                df = download_and_read(key)
                if df is not None and not df.empty:
                    all_dfs.append(df)
                    file_count += 1

        print(f"  -> {file_count} daily files found")

    if not all_dfs:
        return pd.DataFrame()

    return pd.concat(all_dfs, ignore_index=True)


def filter_locations(df: pd.DataFrame,
                      target_start: str = "2022-01-01",
                      target_end: str = "2025-12-31") -> pd.DataFrame:
    """Keep only CPCB/caaqm-network stations whose reporting window actually
    overlaps our target training period. Run this BEFORE bulk downloading -
    it's the difference between downloading ~600 useful stations vs wasting
    time on ~150 irrelevant or dead ones."""
    before = len(df)

    # 1. Provider filter - CPCB and caaqm are the same underlying network,
    # tagged differently depending on submission batch. Drop everything else
    # (low-cost sensor networks, embassy monitors, unidentified sources).
    df = df[df["provider"].isin(["CPCB", "caaqm"])]
    print(f"After provider filter: {len(df)} (from {before})")

    # 2. Drop stations with no reported coverage window at all
    df = df.dropna(subset=["datetime_first", "datetime_last"])
    print(f"After dropping no-coverage-data stations: {len(df)}")

    # 3. Keep only stations whose [first, last] window overlaps our target
    # period - i.e. exclude stations that died before target_start, or only
    # came online after target_end
    df["datetime_first"] = pd.to_datetime(df["datetime_first"], utc=True)
    df["datetime_last"] = pd.to_datetime(df["datetime_last"], utc=True)
    target_start_ts = pd.Timestamp(target_start, tz="UTC")
    target_end_ts = pd.Timestamp(target_end, tz="UTC")

    df = df[(df["datetime_last"] >= target_start_ts) & (df["datetime_first"] <= target_end_ts)]
    print(f"After date-overlap filter ({target_start} to {target_end}): {len(df)}")

    # 4. Drop stations with missing coordinates - can't spatial-join these later
    df = df.dropna(subset=["latitude", "longitude"])
    print(f"After dropping missing-coordinates stations: {len(df)}")

    return df


if __name__ == "__main__":
    locations_df = pd.read_csv(LOCATIONS_PATH)
    print(f"Loaded {len(locations_df)} total India locations")

    locations_df = filter_locations(locations_df)

    # Save the filtered list too, so you have a record of exactly which
    # stations went into your dataset - useful for your project report
    filtered_path = LOCATIONS_PATH.parent / "openaq_india_locations_filtered.csv"
    locations_df.to_csv(filtered_path, index=False)
    print(f"Saved filtered station list to {filtered_path}")

    location_ids = locations_df["location_id"].dropna().astype(int).unique().tolist()
    print(f"Downloading data for {len(location_ids)} locations, years {YEARS}")

    result_df = bulk_download(location_ids)

    if not result_df.empty:
        result_df.to_csv(OUT_PATH, index=False)
        print(f"\nDone. Saved {len(result_df)} rows to {OUT_PATH}")
    else:
        print("\nNo data found - check location_ids and year range.")

# ---------------------------------------------------------------------------
# EXPECTED RAW COLUMNS (verify against your actual downloaded file - OpenAQ's
# CSV schema has evolved over versions): typically includes location_id,
# sensor_id, parameter, value, unit, datetime (UTC), latitude, longitude.
#
# NEXT STEP: pivot from long format (one row per parameter reading) to wide
# format (one row per location-date with a column per pollutant) before
# merging with your satellite/weather data - similar to the CPCB pivot note
# in merge_and_preprocess.py.
# ---------------------------------------------------------------------------


Loaded 758 total India locations
After provider filter: 649 (from 758)
After dropping no-coverage-data stations: 647
After date-overlap filter (2022-01-01 to 2025-12-31): 585
After dropping missing-coordinates stations: 585
Saved filtered station list to data/raw/openaq_india_locations_filtered.csv
[1/585] location_id=17
  -> 312 daily files found
[2/585] location_id=50
  -> 309 daily files found
[3/585] location_id=235
  -> 315 daily files found
[4/585] location_id=301
  -> 176 daily files found
[5/585] location_id=407
  -> 412 daily files found
[6/585] location_id=697
  -> 104 daily files found
[7/585] location_id=860
  -> 434 daily files found
[8/585] location_id=2456
  -> 416 daily files found
[9/585] location_id=2583
  -> 225 daily files found
[10/585] location_id=2586
  -> 396 daily files found
[11/585] location_id=2596
  -> 231 daily files found
[12/585] location_id=5404
  -> 302 daily files found
[13/585] location_id=5408
  -> 312 daily files found
[14/585] location_id=5443
  -

In [ ]:
# Zip the entire folder
!zip -r cpcb_dataset_archive.zip /content/data


Streaming output truncated to the last 5000 lines.
  adding: content/data/raw/openaq_s3_cache/records_csv.gz_locationid=3409387_year=2025_month=07_location-3409387-20250725.csv.gz (deflated 3%)
  adding: content/data/raw/openaq_s3_cache/records_csv.gz_locationid=6923_year=2025_month=10_location-6923-20251002.csv.gz (stored 0%)
  adding: content/data/raw/openaq_s3_cache/records_csv.gz_locationid=3409508_year=2025_month=06_location-3409508-20250604.csv.gz (deflated 4%)
  adding: content/data/raw/openaq_s3_cache/records_csv.gz_locationid=6969_year=2025_month=10_location-6969-20251013.csv.gz (deflated 3%)
  adding: content/data/raw/openaq_s3_cache/records_csv.gz_locationid=228403_year=2022_month=06_location-228403-20220603.csv.gz (stored 0%)
  adding: content/data/raw/openaq_s3_cache/records_csv.gz_locationid=6947_year=2022_month=06_location-6947-20220602.csv.gz (stored 0%)
  adding: content/data/raw/openaq_s3_cache/records_csv.gz_locationid=11579_year=2022_month=01_location-11579-20220128

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Copy the zip file to the root of your Google Drive
!cp /content/cpcb_dataset_archive.zip /content/drive/MyDrive/


In [ ]:
import pandas as pd

# Load just the top 5 rows to verify the data structure
df = pd.read_csv('/content/data/raw/openaq_s3_cache/records_csv.gz_locationid=10484_year=2022_month=01_location-10484-20220101.csv.gz', nrows=5)
print(df)


   location_id  sensors_id                                location  \
0        10484       34781  Sri Aurobindo Marg, Delhi - DPCC-10484   
1        10484       34781  Sri Aurobindo Marg, Delhi - DPCC-10484   
2        10484       34781  Sri Aurobindo Marg, Delhi - DPCC-10484   
3        10484       34772  Sri Aurobindo Marg, Delhi - DPCC-10484   
4        10484       34772  Sri Aurobindo Marg, Delhi - DPCC-10484   

                    datetime        lat        lon parameter  units  value  
0  2022-01-01T04:45:00+05:30  28.531346  77.190156      pm10  µg/m³  240.0  
1  2022-01-01T16:00:00+05:30  28.531346  77.190156      pm10  µg/m³  257.0  
2  2022-01-01T23:00:00+05:30  28.531346  77.190156      pm10  µg/m³  295.0  
3  2022-01-01T04:45:00+05:30  28.531346  77.190156      pm25  µg/m³  184.0  
4  2022-01-01T16:00:00+05:30  28.531346  77.190156      pm25  µg/m³  190.0  


In [ ]:
df.head(10)

,location_id,sensors_id,location,datetime,lat,lon,parameter,units,value
0,10484,34781,"Sri Aurobindo Marg, Delhi - DPCC-10484",2022-01-01T04:45:00+05:30,28.531346,77.190156,pm10,µg/m³,240.0
1,10484,34781,"Sri Aurobindo Marg, Delhi - DPCC-10484",2022-01-01T16:00:00+05:30,28.531346,77.190156,pm10,µg/m³,257.0
2,10484,34781,"Sri Aurobindo Marg, Delhi - DPCC-10484",2022-01-01T23:00:00+05:30,28.531346,77.190156,pm10,µg/m³,295.0
3,10484,34772,"Sri Aurobindo Marg, Delhi - DPCC-10484",2022-01-01T04:45:00+05:30,28.531346,77.190156,pm25,µg/m³,184.0
4,10484,34772,"Sri Aurobindo Marg, Delhi - DPCC-10484",2022-01-01T16:00:00+05:30,28.531346,77.190156,pm25,µg/m³,190.0


In [ ]:
print(df.columns.tolist())
print(df.head(10))

['location_id', 'sensors_id', 'location', 'datetime', 'lat', 'lon', 'parameter', 'units', 'value']
   location_id  sensors_id                                location  \
0        10484       34781  Sri Aurobindo Marg, Delhi - DPCC-10484   
1        10484       34781  Sri Aurobindo Marg, Delhi - DPCC-10484   
2        10484       34781  Sri Aurobindo Marg, Delhi - DPCC-10484   
3        10484       34772  Sri Aurobindo Marg, Delhi - DPCC-10484   
4        10484       34772  Sri Aurobindo Marg, Delhi - DPCC-10484   

                    datetime        lat        lon parameter  units  value  
0  2022-01-01T04:45:00+05:30  28.531346  77.190156      pm10  µg/m³  240.0  
1  2022-01-01T16:00:00+05:30  28.531346  77.190156      pm10  µg/m³  257.0  
2  2022-01-01T23:00:00+05:30  28.531346  77.190156      pm10  µg/m³  295.0  
3  2022-01-01T04:45:00+05:30  28.531346  77.190156      pm25  µg/m³  184.0  
4  2022-01-01T16:00:00+05:30  28.531346  77.190156      pm25  µg/m³  190.0  


In [ ]:
import pandas as pd
import glob
from pathlib import Path
import zipfile
import os
import gc # Import garbage collector for explicit memory management

# Original OUT_PATH points to the combined file; now it'll be used for the parent directory.
OUT_PATH = Path("./data/raw/cpcb_daily_wide.csv")

# Define YEARS within this cell for self-containment
YEARS = [2022, 2023, 2024, 2025]

def load_all_files(cache_dir: Path, target_year: int, batch_size: int = 2000) -> pd.DataFrame:
    # Filter files specifically for the target_year
    files = glob.glob(str(cache_dir / f"*year={target_year}*.csv.gz"))
    print(f"Found {len(files)} cached CSV files for year {target_year}.")

    all_combined_dfs_for_final_concat = [] # List to hold the results of concatenating each batch
    bad_files = 0

    if not files: # Handle case where no files are found for the year
        print(f"No files found for year {target_year}. Returning empty DataFrame.")
        return pd.DataFrame()

    num_batches = (len(files) + batch_size - 1) // batch_size # Calculate total number of batches

    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, len(files))
        batch_files = files[start_idx:end_idx]

        current_batch_dfs = [] # List to hold dataframes for the current batch
        print(f"Processing batch {batch_idx + 1}/{num_batches} for year {target_year} (files {start_idx + 1} to {end_idx})...")

        for f in batch_files:
            try:
                df = pd.read_csv(f)
                if not df.empty:
                    current_batch_dfs.append(df)
            except Exception as e:
                print(f"  Error reading file {f}: {e}")
                bad_files += 1

        if current_batch_dfs:
            batch_combined_df = pd.concat(current_batch_dfs, ignore_index=True)
            all_combined_dfs_for_final_concat.append(batch_combined_df)
            print(f"  Batch {batch_idx + 1} for year {target_year} combined successfully. Rows in this batch: {len(batch_combined_df)}.")
            del current_batch_dfs # Explicitly release memory for this batch's individual dataframes
            gc.collect() # Trigger garbage collection
        else:
            print(f"  No valid dataframes found in batch {batch_idx + 1} for year {target_year}.")

    if bad_files:
        print(f"Skipped {bad_files} unreadable/corrupted files for year {target_year}.")

    if not all_combined_dfs_for_final_concat:
        print(f"No dataframes to concatenate for year {target_year}. Returning empty DataFrame.")
        return pd.DataFrame()

    combined = pd.concat(all_combined_dfs_for_final_concat, ignore_index=True)
    print(f"Combined: {len(combined)} total readings across all files for year {target_year}.")
    return combined


def clean_and_pivot(df: pd.DataFrame) -> pd.DataFrame:
    # Handle case where df might be empty after load_all_files returns empty DataFrame
    if df.empty:
        print("Input DataFrame for clean_and_pivot is empty. Returning empty DataFrame.")
        return pd.DataFrame()

    # Parse datetime - already carries +05:30 IST offset, keep as-is and just
    # extract the calendar date for daily aggregation
    # Using utc=True and errors='coerce' to robustly convert to datetime and handle mixed timezones
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors='coerce')
    df["date"] = df["datetime"].dt.date

    # Keep one lat/lon/location-name per location_id (static per station,
    # take the first occurrence rather than carrying it through every row)
    station_meta = (
        df[["location_id", "location", "lat", "lon"]]
        .drop_duplicates(subset=["location_id"])
        .rename(columns={"location": "station_name", "lat": "latitude", "lon": "longitude"})
    )

    # Daily average per station-date-parameter (collapses the 4-10 intraday
    # readings you saw per file into one number per pollutant per day)
    daily = (
        df.groupby(["location_id", "date", "parameter"])[["value"]]
        .mean()
        .reset_index()
    )

    # Pivot: one row per station-date, one column per pollutant
    wide = daily.pivot_table(
        index=["location_id", "date"],
        columns="parameter",
        values="value",
    ).reset_index()
    wide.columns.name = None  # drop the pivot's column-index name artifact

    # Reattach station metadata (name, coordinates)
    wide = wide.merge(station_meta, on="location_id", how="left")

    # Rename to match what merge_and_preprocess.py expects downstream
    wide = wide.rename(columns={"location_id": "station_id"})

    return wide


if __name__ == "__main__":
    # Define the path to the zipped archive in Google Drive
    ZIP_ARCHIVE_PATH = Path("/content/drive/MyDrive/major project dataset/CPCB/cpcb_dataset_archive.zip")
    # Define the directory where the contents will be extracted
    EXTRACT_TO_DIR = Path("./extracted_cpcb_data")

    # Create the extraction directory if it doesn't exist
    EXTRACT_TO_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Extracting {ZIP_ARCHIVE_PATH} to {EXTRACT_TO_DIR}...")
    try:
        with zipfile.ZipFile(ZIP_ARCHIVE_PATH, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_TO_DIR)
        print("Extraction complete.")
    except FileNotFoundError:
        print(f"Error: ZIP archive not found at {ZIP_ARCHIVE_PATH}. Please ensure the file exists.")
        exit() # Exit if the zip file isn't found
    except Exception as e:
        print(f"An error occurred during extraction: {e}")
        exit()

    # Update ACTUAL_CACHE_DIR to point to the actual directory containing the csv.gz files
    # Based on how the zip was created (`!zip -r cpcb_dataset_archive.zip /content/data`),
    # the files will be inside `EXTRACT_TO_DIR / "content" / "data" / "raw" / "openaq_s3_cache"`.
    ACTUAL_CACHE_DIR = EXTRACT_TO_DIR / "content" / "data" / "raw" / "openaq_s3_cache"

    for year in YEARS:
        print(f"\n--- Processing data for year: {year} ---")
        raw_year = load_all_files(ACTUAL_CACHE_DIR, target_year=year, batch_size=2000)
        wide_year = clean_and_pivot(raw_year)

        # Define a year-specific output path
        OUT_PATH_YEAR = OUT_PATH.parent / f"cpcb_daily_wide_{year}.csv"

        if not wide_year.empty:
            OUT_PATH_YEAR.parent.mkdir(parents=True, exist_ok=True)
            wide_year.to_csv(OUT_PATH_YEAR, index=False)

            print(f"\nSaved {len(wide_year)} station-date rows for {year} to {OUT_PATH_YEAR}")
            print(f"Pollutant columns found for {year}: {[c for c in wide_year.columns if c not in ['station_id', 'date', 'station_name', 'latitude', 'longitude']]}")
            print(f"\nMissing value summary for {year}:\n{wide_year.isnull().sum()}")
            print(f"\nDate range for {year}: {wide_year['date'].min()} to {wide_year['date'].max()}")
            print(f"Unique stations for {year}: {wide_year['station_id'].nunique()}")
        else:
            print(f"\nNo data to save after cleaning and pivoting for year {year}.")

        # Explicitly release memory and trigger garbage collection after each year
        del raw_year, wide_year
        gc.collect()

Extracting /content/drive/MyDrive/major project dataset/CPCB/cpcb_dataset_archive.zip to extracted_cpcb_data...
Extraction complete.

--- Processing data for year: 2022 ---
Found 32745 cached CSV files for year 2022.
Processing batch 1/17 for year 2022 (files 1 to 2000)...
  Batch 1 for year 2022 combined successfully. Rows in this batch: 186703.
Processing batch 2/17 for year 2022 (files 2001 to 4000)...
  Batch 2 for year 2022 combined successfully. Rows in this batch: 189799.
Processing batch 3/17 for year 2022 (files 4001 to 6000)...
  Batch 3 for year 2022 combined successfully. Rows in this batch: 193413.
Processing batch 4/17 for year 2022 (files 6001 to 8000)...
  Batch 4 for year 2022 combined successfully. Rows in this batch: 188934.
Processing batch 5/17 for year 2022 (files 8001 to 10000)...
  Batch 5 for year 2022 combined successfully. Rows in this batch: 191342.
Processing batch 6/17 for year 2022 (files 10001 to 12000)...
  Batch 6 for year 2022 combined successfully. R

- Above code saparate year wise data
- but missing 2023 and 2024 data
- and due to lot of data in 2025, Ram crashes due to limited 12.7 GB ram.